In [1]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser , PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel , Field
from typing import Literal , List

c:\Users\devra\miniconda3\envs\dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\devra\miniconda3\envs\dev\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
load_dotenv()

model = GoogleGenerativeAIEmbeddings(model= 'gemini-embedding-2-preview')

chat_model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [3]:
retriever = WikipediaRetriever(
    top_k_results=2 , 
    lang =  'en'
)

In [4]:
query = "AGENTIC AI"

In [5]:
retrive_doc = retriever.invoke(query)

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, 
    chunk_overlap = 0
)

In [7]:
document = splitter.split_documents(retrive_doc)

In [8]:
print(len(document))

9


In [9]:
for i in range(len(document)):
    print(f'Length of the Documnet  :' , (len(document[i].page_content)))

Length of the Documnet  : 340
Length of the Documnet  : 581
Length of the Documnet  : 596
Length of the Documnet  : 993
Length of the Documnet  : 58
Length of the Documnet  : 984
Length of the Documnet  : 437
Length of the Documnet  : 731
Length of the Documnet  : 994


In [10]:
chroma_store = Chroma(
    embedding_function=model , 
    collection_name = 'sample' , 
    persist_directory='chroma_database'
)

In [11]:
chroma_store.add_documents(document)

['020ed0f6-d4ff-41c7-a69e-e7c614933cc0',
 'ef29b692-88b5-4aec-b2af-83371cfa467f',
 '65cc1116-4c7f-477f-9d91-d21a5f3eaea1',
 '37f403a9-4b70-4597-b218-7c0c58c4e717',
 'b7ce6cd4-f028-4bfa-ac78-e0a6cf49bd5d',
 'f25a08ed-13a9-45f3-a8e8-23740829e557',
 'e66fb9ef-b8f3-4ebc-9c00-6125133d2038',
 'e4aca2d5-8c9e-4067-a493-a7f7fd947c1b',
 '43764b46-75b8-47d8-ab96-8151dceecfbc']

In [12]:
len(chroma_store.get(include = ['embeddings'])['embeddings'][0])

3072

In [13]:
query_1_simi_score = chroma_store.similarity_search_with_score(
    query = 'Langchain', 
    k = 3
)

In [14]:
print(f'The id is : ' , query_1_simi_score[0][0].id)
print(f'The metadata is : ' , query_1_simi_score[0][0].metadata)
print(f'the page content :' , query_1_simi_score[0][0].page_content)
print(f' the highet score for query :' , query_1_simi_score[0][1])

The id is :  eeae4211-da02-40e3-8056-4e9fb34c8264
The metadata is :  {'summary': 'In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents distinguished by their ability to operate autonomously in complex environments. Agentic AI tools prioritize decision-making over content creation and do not require continuous oversight.\n\n', 'title': 'AI agent', 'source': 'https://en.wikipedia.org/wiki/AI_agent'}
the page content : Retrieval-augmented generation
ReAct (Reason + Act) pattern is an iterative process in which an AI agent alternates between reasoning and taking actions, receives observations from the environment or external tools, and integrates these observations into subsequent reasoning steps.
Reflexion, which uses an LLM to create feedback on the agent's plan of action and stores that feedback in a memory cache.
A tool/agent registry, for organ
 the highet score for query : 0.674597620

In [15]:
for i in range(len(document)):
    text = document[i].page_content

In [16]:
text

'=== Acquisition by Meta Platforms ===\nIn December 2025, Meta announced that it would acquire Manus, the company behind the Manus AI agent. Meta did not disclose financial terms; reporting suggests the deal is valued between US$2-3 billion. Meta said it would continue to operate and sell the Manus service and integrate its technology into products including Meta AI. Manus said it would continue offering subscriptions through its own app and website and remain based in Singapore. In January 2026, it was announced that Manus was the subject of an initial regulatory review by Chinese authorities following the proposed acquisition. The review focuses on whether artificial intelligence technologies developed by Manus while it was based in China fall under national security or technology export regulations. Beijing subsequently issued exit bans for Manus executives under scrutiny.\n\n\n== References ==\n\n\n== External links ==\nOfficial website \nAgentic AI Platform – Features and Capabili

In [17]:
len(text)

994

In [18]:
prompt_1 = PromptTemplate(
    template='Generate the best summary from the following text \n {text}' , 
    input_variables=['text']
)

In [19]:
parser = StrOutputParser()

In [20]:
summary_chain = prompt_1 | chat_model | parser 

In [44]:
class final(BaseModel):

    key_point : str =Field(description='Extract the 5 key points from the Given Summary')

    top_key_words : str = Field(description="Generate the best key words that the entire summary represents")

In [ ]:
pydantic_parser = PydanticOutputParser(pydantic_object=final)

In [48]:
prompt_2 = PromptTemplate(
    template='From the Summary extract the key points and a top key words from summary \n{summary} \n{format_instruction}' , 
    input_variables= ['summary'] , 
    partial_variables={'format_instruction': pydantic_parser.get_format_instructions()}  
    )

In [49]:
chain_2 = prompt_2 | chat_model  | pydantic_parser

In [50]:
final_chain = summary_chain | chain_2 

result = final_chain.invoke({'text' : text})

In [55]:
print(result.key_point)
print(result.top_key_words)

1. In December 2025, Meta acquired Singapore-based AI firm Manus for $2–3 billion. 2. Meta intends to integrate Manus’s technology into its AI ecosystem while keeping existing subscriptions. 3. Chinese authorities initiated a regulatory investigation in January 2026. 4. The probe is focused on potential violations of national security and export control regulations. 5. The investigation has led to travel exit bans being imposed on several Manus executives.
Meta, Manus, AI acquisition, regulatory review, national security, export regulations
